# NB4 · Publish artifacts to Hugging Face

**Team Entity:** Sourabh Dahiya, Amritanshu Singh, Sidhant Mahi

This notebook publishes the public outputs of NB1–NB3 for the Lexsi AI Safety and Alignment Hackathon (Track 2, Problem #3: multi-turn audits).

It creates:
1. a **public dataset repo** with the scenario suite, redacted transcripts, metric tables, run manifest and figures;
2. a **public model repo** with the refusal direction and steering configuration used in NB3 (no weights were fine-tuned; the base model is unchanged);
3. two **public collections**, one for models and one for datasets, as required by the submission form.

Unredacted model outputs (`private_full/`) and the superseded any-turn scores (`anyturn_backup/`) are never uploaded.

**Requirements:** a CPU runtime is enough. The `HF_TOKEN` Colab secret must have **write** permission.

## 1 · Connect to Google Drive and Hugging Face

Mounts Drive, where NB1–NB3 saved their results, logs in to Hugging Face with the token stored in Colab secrets, and reads your username so every repo is created under your account.

In [ ]:
from pathlib import Path
from google.colab import drive, userdata
from huggingface_hub import HfApi, login, create_collection, add_collection_item

drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/mta_v3')
RUN_ID = 'v3-aya8b'

login(userdata.get('HF_TOKEN'))
api = HfApi()
user = api.whoami()['name']
print('HF user:', user, '| results folder:', WORK)

## 2 · Choose what is public

Builds the exact list of files that will be uploaded: the scenario suite, every metric CSV, the redacted transcripts and the export folder (manifest, figures, redacted after/control transcripts).

Anything under `private_full/` or `anyturn_backup/` is excluded, and so is `export/transcripts_before__*.jsonl`, which holds the superseded any-turn scores (the re-scored baseline is in `transcripts/`). The cell stops with an error if such a file slips into the list, or if any file contains what looks like a Hugging Face token.

In [ ]:
PRIVATE = ('private_full', 'anyturn_backup')

public = [WORK / 'scenarios.json']
public += sorted(WORK.glob('*.csv'))
public += sorted((WORK / 'transcripts').glob('*.json'))
public += sorted(p for p in (WORK / 'export').rglob('*') if p.is_file() and not p.name.startswith('transcripts_before__'))

assert not any(part in PRIVATE for p in public for part in p.relative_to(WORK).parts), 'private file in upload list'
for p in public:
    if p.suffix in ('.json', '.jsonl', '.csv', '.txt', '.md'):
        assert 'hf_' not in p.read_text(errors='ignore'), f'possible token in {p.name}'

print(len(public), 'files will be published')
for p in public[:8]:
    print('  ', p.relative_to(WORK))
print('   ...')

## 3 · Publish the dataset

Creates the public dataset repo and uploads the files selected above, keeping the same folder layout as on Drive so the notebooks' paths still match. A short dataset card explains the contents and the redaction.

In [ ]:
DS = f'{user}/aya-expanse-8b-multiturn-audit'
api.create_repo(DS, repo_type='dataset', private=False, exist_ok=True)

for p in public:
    api.upload_file(path_or_fileobj=str(p), path_in_repo=str(p.relative_to(WORK)),
                    repo_id=DS, repo_type='dataset')

card = f'''---
license: cc-by-nc-4.0
tags: [ai-safety, red-teaming, multi-turn, aya-expanse]
---
# Multi-turn safety audit of Aya Expanse 8B

Artifacts from a multi-turn safety audit of `CohereLabs/aya-expanse-8b` (Lexsi AI Safety and Alignment Hackathon, Track 2, Problem #3).

- `scenarios.json`: 66 multi-turn scenarios (50 attacks in five families, 16 benign controls) with dev/test split and payload turn.
- `transcripts/`: per-scenario records for every run (baseline, dev sweep, test, random control). Responses are truncated to 300 characters and hashed.
- `*.csv`: metric tables (baseline, turn of break, single vs multi turn, sweep, before/after).
- `export/`: run manifest, figures and redacted after/control transcripts.

Primary scoring is payload-gated: a break counts only at or after the scenario's payload turn. `base_metrics`, `transcripts/` and all NB3 outputs use it; `turn_of_break` and `single_vs_multi` are NB2 tables computed with any-turn scoring. Run id `{RUN_ID}`.
Full, unredacted model outputs are intentionally not published.
'''
api.upload_file(path_or_fileobj=card.encode(), path_in_repo='README.md', repo_id=DS, repo_type='dataset')
print('dataset: https://huggingface.co/datasets/' + DS)

## 4 · Publish the repair artifact

The repair in NB3 is activation steering, not fine-tuning, so the artifact is the refusal direction plus its settings. This uploads the direction file(s) and the run manifest (layer, chosen strength, seed, dev sweep and test results) to a public model repo with a short model card.

In [ ]:
MD = f'{user}/aya-expanse-8b-refusal-direction'
api.create_repo(MD, repo_type='model', private=False, exist_ok=True)

directions = [p for p in WORK.rglob('*.pt') if not any(x in p.parts for x in PRIVATE)]
print('direction files:', [p.name for p in directions])
assert directions, 'no direction file found: check where NB3 cached it'

for p in directions:
    api.upload_file(path_or_fileobj=str(p), path_in_repo=p.name, repo_id=MD)
api.upload_file(path_or_fileobj=str(WORK / 'export' / f'manifest__{RUN_ID}.json'),
                path_in_repo='manifest.json', repo_id=MD)

card = '''---
base_model: CohereLabs/aya-expanse-8b
license: cc-by-nc-4.0
tags: [ai-safety, activation-steering, refusal-direction]
---
# Refusal direction for Aya Expanse 8B

A refusal direction extracted with SafeTune from `CohereLabs/aya-expanse-8b` and used for activation steering in a multi-turn safety audit.
No model weights were changed; apply the direction to the base model at the layer and strength given in `manifest.json`.

Steering strength was selected on the dev split only. On the held-out test split it did not reduce attack success; see the manifest and the linked dataset for full results.
'''
api.upload_file(path_or_fileobj=card.encode(), path_in_repo='README.md', repo_id=MD)
print('model repo: https://huggingface.co/' + MD)

## 5 · Create the collections for the submission form

The form asks for one public model collection and one public dataset collection. The model collection holds the base model and the refusal-direction repo; the dataset collection holds the audit dataset. The two links printed at the end go into the form.

In [ ]:
mc = create_collection(title='Multi-turn audit of Aya Expanse 8B: models', namespace=user, private=False, exists_ok=True)
add_collection_item(mc.slug, item_id='CohereLabs/aya-expanse-8b', item_type='model', exists_ok=True)
add_collection_item(mc.slug, item_id=MD, item_type='model', exists_ok=True)

dc = create_collection(title='Multi-turn audit of Aya Expanse 8B: data', namespace=user, private=False, exists_ok=True)
add_collection_item(dc.slug, item_id=DS, item_type='dataset', exists_ok=True)

print('MODEL COLLECTION:   https://huggingface.co/collections/' + mc.slug)
print('DATASET COLLECTION: https://huggingface.co/collections/' + dc.slug)

## 6 · Check that everything is public

Lists the published repos and confirms none is private or gated, which the rules require. Open both collection links in a logged-out browser window as a final check.

In [ ]:
for rid, rtype in [(DS, 'dataset'), (MD, 'model')]:
    info = api.repo_info(rid, repo_type=rtype)
    files = api.list_repo_files(rid, repo_type=rtype)
    leaked = [f for f in files if f.startswith(PRIVATE)]
    print(f'{rid}: private={info.private} gated={getattr(info, "gated", False)} files={len(files)} private_files={leaked}')
    assert not info.private and not getattr(info, 'gated', False) and not leaked